# 2026-cv-competition — детекция (52 класса)

Разделы: **0. Данные (опц.)** → **EDA** → **обучение** (YOLOv8, разметка уже YOLO) → **инференс** → **постобработка** → **`sample_submission.csv`**.

Запускайте из корня репозитория (рядом с папкой `2026-cv-competition`). Зависимости: `uv sync`, затем ядро из этого окружения (`uv run python -m ipykernel install --user --name cv-hw-2026`).

**Seed везде: 993.**

**Сабмит:** весь пайплайн в одном .ipynb (без отдельных .py). Зависимости подхватывает **следующая** ячейка с `pip` — затем **Run all**.


In [ ]:
# Kaggle / чистая среда: пакеты (если уже есть — сработает быстро)
import subprocess
import sys

subprocess.check_call(
    [sys.executable, "-m", "pip", "install", "-q", "ultralytics", "kagglehub"],
)


## 0. Где лежат данные (Kaggle / kagglehub)

- **Обычный случай (локальный репо):** папка `2026-cv-competition` рядом с ноутбуком — **ячейку ниже не запускай** (или она ничего не сделает).
- **Kaggle, есть Add data:** данные в `/kaggle/input/...` — ячейка ниже **подхватит** путь, если `sample_submission.csv` внутри `.../2026-cv-competition/`.
- **Нет Add data:** поставьте `USE_KAGGLEHUB = True` **или** `USE_KAGGLEHUB=1` в среде; нужны права на соревнование и [Kaggle API](https://www.kaggle.com/docs/api) (`kaggle.json` / Settings → API).
- **Вручную:** `SOLUTION_ROOT` = каталог, внутри которого лежит `2026-cv-competition/sample_submission.csv` (например `"/kaggle/working/имя-репо"`).


In [ ]:
import os
from pathlib import Path

# True — скачать соревнование через kagglehub (нужен пакет kagglehub + доступ к соревнованию)
USE_KAGGLEHUB = os.environ.get("USE_KAGGLEHUB", "0") == "1"  # либо сюда: USE_KAGGLEHUB = True
KAGGLE_COMP = os.environ.get("KAGGLE_COMP", "2026-cv-competition")


def _has_data(root: Path) -> bool:
    return (root / "2026-cv-competition" / "sample_submission.csv").is_file()


def _chdir_to_kaggle_input() -> bool:
    base = Path("/kaggle/input")
    if not base.is_dir():
        return False
    for sub in sorted(base.iterdir()):
        if not sub.is_dir():
            continue
        if _has_data(sub):
            os.chdir(sub)
            return True
    return False


def _chdir_from_kagglehub() -> None:
    import kagglehub  # noqa: WPS433

    # как в доке: скачать последнюю версию
    path = kagglehub.competition_download(KAGGLE_COMP)
    print("Path to competition files:", path)
    p = Path(path).resolve()
    for c in (p, p.parent):
        if _has_data(c):
            os.chdir(c)
            return
    if p.is_dir() and p.name == "2026-cv-competition" and (p / "sample_submission.csv").is_file():
        os.chdir(p.parent)
        return
    raise FileNotFoundError(
        "kagglehub: нет 2026-cv-competition/sample_submission.csv рядом с " + str(p)
    )


_sr = os.environ.get("SOLUTION_ROOT", "").strip()
if _sr:
    r = Path(_sr).resolve()
    if not _has_data(r):
        raise FileNotFoundError("SOLUTION_ROOT: ожидается .../2026-cv-competition/sample_submission.csv, путь: " + str(r))
    os.chdir(r)
elif USE_KAGGLEHUB:
    _chdir_from_kagglehub()
elif _chdir_to_kaggle_input():
    pass

print("Рабочая папка:", Path.cwd().resolve())


In [ ]:
import os
import random
from pathlib import Path

import numpy as np
import pandas as pd

RNG_SEED = 993

os.environ["PYTHONHASHSEED"] = str(RNG_SEED)
random.seed(RNG_SEED)
np.random.seed(RNG_SEED)

import torch

torch.manual_seed(RNG_SEED)
torch.cuda.manual_seed_all(RNG_SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

print("torch:", torch.__version__, "cuda:", torch.cuda.is_available())

In [ ]:
import os
from pathlib import Path

# Возможные раскладки относительно корня соревнования (train / test)
_LAYOUTS = [
    ("train/train/images", "train/train/labels", "test/test/images"),
    ("train/images", "train/labels", "test/images"),
]


def _valid_comp(d: Path) -> bool:
    if not (d / "sample_submission.csv").is_file():
        return False
    for ti, tl, tst in _LAYOUTS:
        if (d / ti).is_dir() and (d / tl).is_dir() and (d / tst).is_dir():
            return True
    return False


def _pick_paths(comp: Path):
    for ti, tl, tst in _LAYOUTS:
        if (comp / ti).is_dir() and (comp / tl).is_dir() and (comp / tst).is_dir():
            return comp / ti, comp / tl, comp / tst
    return None


def _find_comp_dir() -> Path:
    _env = os.environ.get("SOLUTION_ROOT", "").strip()
    if _env:
        r = Path(_env).resolve()
        for d in (r / "2026-cv-competition", r):
            if _valid_comp(d):
                return d.resolve()
    here = Path.cwd().resolve()
    for p in [here, *here.parents]:
        d = p / "2026-cv-competition"
        if _valid_comp(d):
            return d.resolve()
        if _valid_comp(p):
            return p.resolve()
    kin = Path("/kaggle/input")
    hints = []
    if kin.is_dir():
        for cand in kin.rglob("sample_submission.csv"):
            hints.append(str(cand))
            par = cand.parent
            for d in (par, par.parent, par.parent.parent):
                if d.is_dir() and _valid_comp(d):
                    return d.resolve()
    msg = (
        "Не найдены данные соревнования: нужны sample_submission.csv и папки train/test.\n"
        "Сделайте: (1) Add data → соревнование; (2) ячейка «0» + Internet + kagglehub; "
        "(3) SOLUTION_ROOT на каталог, где лежит sample_submission и train/.\n"
    )
    if hints:
        msg += "Найденные sample_submission.csv (до 8 шт.): " + repr(hints[:8])
    else:
        msg += "Файл sample_submission.csv под /kaggle/input не найден — датасет не подключён."
    raise FileNotFoundError(msg)


def _find_workspace_root() -> Path:
    if Path("/kaggle/working").is_dir():
        return Path("/kaggle/working").resolve()
    here = Path.cwd().resolve()
    for p in [here, *here.parents]:
        if (p / "2026-cv-competition" / "sample_submission.csv").is_file():
            return p
    return here


COMP = _find_comp_dir()
ROOT = _find_workspace_root()

_paths = _pick_paths(COMP)
if _paths is None:
    raise FileNotFoundError("COMP найден, но нет ни одной известной схемы train/labels/test: " + str(COMP))
TRAIN_IMG, TRAIN_LBL, TEST_IMG = _paths
SAMPLE_SUB = COMP / "sample_submission.csv"

RUNS = ROOT / "runs"
RUNS.mkdir(parents=True, exist_ok=True)

for p in (TRAIN_IMG, TRAIN_LBL, TEST_IMG):
    assert p.is_dir(), p
print("ROOT (рабочая папка):", ROOT)
print("COMP (данные соревнования):", COMP)
print("TRAIN_IMG:", TRAIN_IMG)


## 1. EDA

In [ ]:
from collections import Counter

import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.patches import Rectangle
from PIL import Image

sns.set_theme(style="whitegrid", context="notebook")

label_files = sorted(TRAIN_LBL.glob("*.txt"))
class_counts = Counter()
objs_per_image = []

for lf in label_files:
    lines = [ln for ln in lf.read_text(encoding="utf-8").splitlines() if ln.strip()]
    objs_per_image.append(len(lines))
    for ln in lines:
        class_counts[int(ln.split()[0])] += 1

n_classes = max(class_counts) + 1 if class_counts else 0
print("Размеченных файлов:", len(label_files))
print("Число классов (max id + 1):", n_classes)
print("Всего объектов:", sum(class_counts.values()))
print("Объектов на картинку: min/median/max", np.min(objs_per_image), np.median(objs_per_image), np.max(objs_per_image))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

cls_ids = sorted(class_counts)
axes[0].bar(cls_ids, [class_counts[c] for c in cls_ids], color="steelblue")
axes[0].set_title("Частота классов (все боксы)")
axes[0].set_xlabel("class_id")
axes[0].set_ylabel("count")

axes[1].hist(objs_per_image, bins=30, color="coral", edgecolor="white")
axes[1].set_title("Число объектов на изображение")
axes[1].set_xlabel("objects per image")
plt.tight_layout()
plt.show()

In [ ]:
def yolo_norm_to_xyxy(cx, cy, w, h):
    return cx - w / 2, cy - h / 2, cx + w / 2, cy + h / 2


def show_annotated(img_path: Path):
    img = Image.open(img_path).convert("RGB")
    w0, h0 = img.size
    lf = TRAIN_LBL / (img_path.stem + ".txt")
    fig, ax = plt.subplots(1, 1, figsize=(8, 8))
    ax.imshow(img)
    ax.axis("off")
    if lf.is_file():
        for ln in lf.read_text(encoding="utf-8").splitlines():
            if not ln.strip():
                continue
            p = ln.split()
            cid = int(p[0])
            cx, cy, bw, bh = map(float, p[1:5])
            x1, y1, x2, y2 = yolo_norm_to_xyxy(cx, cy, bw, bh)
            x1, x2 = x1 * w0, x2 * w0
            y1, y2 = y1 * h0, y2 * h0
            rect = Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, edgecolor="lime", linewidth=2)
            ax.add_patch(rect)
            ax.text(x1, max(0, y1 - 4), str(cid), color="yellow", fontsize=8, fontweight="bold")
    plt.title(img_path.name)
    plt.show()


demo_imgs = sorted(TRAIN_IMG.glob("*.jpg"))[:3]
for ip in demo_imgs:
    show_annotated(ip)

## 2. Подготовка данных для YOLO (train/val split)

Списки абсолютных путей к изображениям — Ultralytics подставит `labels/` вместо `images/` для разметки.

In [ ]:
from sklearn.model_selection import train_test_split
import yaml

all_images = sorted(TRAIN_IMG.glob("*.jpg"))
train_paths, val_paths = train_test_split(
    all_images,
    test_size=0.15,
    random_state=RNG_SEED,
    shuffle=True,
)

split_dir = RUNS / "yolo_split"
split_dir.mkdir(parents=True, exist_ok=True)
train_txt = split_dir / "train.txt"
val_txt = split_dir / "val.txt"

train_txt.write_text("\n".join(str(p.resolve()) for p in train_paths), encoding="utf-8")
val_txt.write_text("\n".join(str(p.resolve()) for p in val_paths), encoding="utf-8")

names = [str(i) for i in range(52)]
data_yaml = {
    "train": str(train_txt.resolve()),
    "val": str(val_txt.resolve()),
    "nc": 52,
    "names": names,
}
yaml_path = split_dir / "dataset.yaml"
yaml_path.write_text(yaml.safe_dump(data_yaml, sort_keys=False, allow_unicode=True), encoding="utf-8")

print("train:", len(train_paths), "val:", len(val_paths))
print("yaml:", yaml_path)

## 3. Обучение модели (YOLOv8)

Подберите `epochs` и `batch` под GPU/время. На CPU лучше уменьшить `epochs`/`imgsz` или запустить на Kaggle GPU.

По умолчанию Ultralytics пишет `weights/best.pt` и `weights/last.pt`. Дополнительно: `epochN.pt` каждые `SAVE_PERIOD` эпох (см. ячейку кода).


In [ ]:
from ultralytics import YOLO

EPOCHS = int(os.environ.get("EPOCHS", "50"))
IMGSZ = int(os.environ.get("IMGSZ", "640"))
BATCH = int(os.environ.get("BATCH", "16" if torch.cuda.is_available() else "4"))
BASE_MODEL = os.environ.get("BASE_MODEL", "yolov8s.pt")
RUN_NAME = os.environ.get("RUN_NAME", "yolov8s_52cls")
# 0 = только best/last; N>0 = доп. checkpoint `epochN.pt` каждые N эпох
SAVE_PERIOD = int(os.environ.get("SAVE_PERIOD", "5"))
SKIP_TRAIN = os.environ.get("SKIP_TRAIN", "0") == "1"

if SKIP_TRAIN:
    best_ckpt = RUNS / "detect" / RUN_NAME / "weights" / "best.pt"
    assert best_ckpt.is_file(), f"SKIP_TRAIN=1, но нет {best_ckpt}"
    print("skip training, use", best_ckpt)
else:
    model = YOLO(BASE_MODEL)
    model.train(
        data=str(yaml_path),
        epochs=EPOCHS,
        imgsz=IMGSZ,
        batch=BATCH,
        seed=RNG_SEED,
        deterministic=True,
        project=str(RUNS / "detect"),
        name=RUN_NAME,
        exist_ok=True,
        save=True,
        save_period=SAVE_PERIOD,
        workers=int(os.environ.get("WORKERS", "0")),
        patience=int(os.environ.get("PATIENCE", "25")),
        cos_lr=os.environ.get("COS_LR", "1") == "1",
    )

    best_ckpt = RUNS / "detect" / RUN_NAME / "weights" / "best.pt"
    assert best_ckpt.is_file(), best_ckpt
    print("best:", best_ckpt)

if os.environ.get("RUN_VAL", "0") == "1":
    eval_m = YOLO(str(best_ckpt))
    eval_m.val(
        data=str(yaml_path),
        imgsz=IMGSZ,
        batch=int(os.environ.get("VAL_BATCH", str(BATCH))),
        seed=RNG_SEED,
    )


In [ ]:
# Постобработка сабмита (self-contained, без внешних .py)
import numpy as np
import torch
from torchvision.ops import nms

EMPTY_PRED = "0 0.000000 0 0 0 0"


def result_to_pred(result):
    b = result.boxes
    if b is None or len(b) == 0:
        return (
            np.zeros((0, 4), dtype=np.float32),
            np.zeros((0,), dtype=np.float32),
            np.zeros((0,), dtype=np.int32),
        )
    return (
        b.xyxy.cpu().numpy().astype(np.float32),
        b.conf.cpu().numpy().astype(np.float32),
        b.cls.cpu().numpy().astype(np.int32),
    )


def apply_score_weights(conf, cls, w: float):
    return conf * float(w), cls


def merge_image_predictions(preds, img_w: float, img_h: float, score_thr: float, nms_iou: float, max_det: int) -> str:
    boxes_all = []
    scores_all = []
    cls_all = []

    for boxes, scores, cls_ids in preds:
        if boxes.size == 0:
            continue
        keep = scores >= score_thr
        if not np.any(keep):
            continue
        boxes_all.append(boxes[keep])
        scores_all.append(scores[keep])
        cls_all.append(cls_ids[keep])

    if not boxes_all:
        return EMPTY_PRED

    boxes = np.concatenate(boxes_all, axis=0)
    scores = np.concatenate(scores_all, axis=0)
    cls_ids = np.concatenate(cls_all, axis=0)

    out_boxes = []
    out_scores = []
    out_cls = []

    for c in np.unique(cls_ids):
        idx = np.where(cls_ids == c)[0]
        b = torch.from_numpy(boxes[idx]).float()
        s = torch.from_numpy(scores[idx]).float()
        keep_idx = nms(b, s, nms_iou).cpu().numpy()
        out_boxes.append(boxes[idx][keep_idx])
        out_scores.append(scores[idx][keep_idx])
        out_cls.append(np.full(len(keep_idx), int(c), dtype=np.int32))

    boxes = np.concatenate(out_boxes, axis=0)
    scores = np.concatenate(out_scores, axis=0)
    cls_ids = np.concatenate(out_cls, axis=0)

    order = np.argsort(-scores)[: max(1, int(max_det))]
    boxes = boxes[order]
    scores = scores[order]
    cls_ids = cls_ids[order]

    boxes[:, [0, 2]] = np.clip(boxes[:, [0, 2]], 0.0, img_w - 1.0)
    boxes[:, [1, 3]] = np.clip(boxes[:, [1, 3]], 0.0, img_h - 1.0)

    parts = []
    for i in range(len(cls_ids)):
        x1, y1, x2, y2 = boxes[i]
        if x2 < x1:
            x1, x2 = x2, x1
        if y2 < y1:
            y1, y2 = y2, y1
        parts += [
            str(int(cls_ids[i])),
            f"{float(scores[i]):.6f}",
            f"{x1:.2f}",
            f"{y1:.2f}",
            f"{x2:.2f}",
            f"{y2:.2f}",
        ]
    return " ".join(parts) if parts else EMPTY_PRED


## 4. Инференс на тесте (мультимасштаб, без лишнего обучения)

- Дополнительные **эпохи** не обязательны: 2+ масштаба на инференсе + слияние NMS (код в ячейке выше).
- По умолчанию `IMGSZ_LIST=704,832` (если не задать `IMGSZ_LIST` / `IMGSZ_PRED` в env).
- `MS_WEIGHTS` — веса по масштабам; `AUGMENT_LAST_ONLY=1` — TTA только на последнем масштабе.
- `CONF_INFER` / `IOU_INFER` — внутри YOLO; `CONF_THR`, `POST_NMS_IOU`, `MAX_DET` — в ячейке сабмита. Разовая **val-метрика** на разбиении: `RUN_VAL=1` в коде секции «3. Обучение».



In [ ]:
from ultralytics import YOLO

if "best_ckpt" not in globals():
    run_name = os.environ.get("RUN_NAME", "yolov8s_52cls")
    best_ckpt = RUNS / "detect" / run_name / "weights" / "best.pt"

_default_imgsz = int(globals().get("IMGSZ", 640))


def _parse_list_int(s: str, default: int) -> list[int]:
    t = s.strip()
    if not t:
        return [default]
    return [int(x.strip()) for x in t.split(",") if x.strip()]


_imgsz_env = (os.environ.get("IMGSZ_LIST") or os.environ.get("IMGSZ_PRED") or "704,832").strip()
IMGSZ_LIST = _parse_list_int(_imgsz_env, _default_imgsz)

CONF_INFER = float(os.environ.get("CONF_INFER", "0.01"))
IOU_INFER = float(os.environ.get("IOU_INFER", "0.65"))
AUGMENT = os.environ.get("AUGMENT", "1") == "1"
AUGMENT_LAST_ONLY = os.environ.get("AUGMENT_LAST_ONLY", "1") == "1"

_ws = [x.strip() for x in os.environ.get("MS_WEIGHTS", "").split(",") if x.strip()]
if not _ws:
    MS_WEIGHTS = [1.0] * len(IMGSZ_LIST)
else:
    MS_WEIGHTS = [float(x) for x in _ws]
    if len(MS_WEIGHTS) == 1 and len(IMGSZ_LIST) > 1:
        MS_WEIGHTS = [MS_WEIGHTS[0]] * len(IMGSZ_LIST)
    if len(MS_WEIGHTS) != len(IMGSZ_LIST):
        raise ValueError(f"MS_WEIGHTS: ожидается {len(IMGSZ_LIST)} весов, задано {MS_WEIGHTS}")

assert best_ckpt.is_file(), f"Нет чекпойнта {best_ckpt} — обучите или SKIP_TRAIN=1"

infer_model = YOLO(str(best_ckpt))
test_paths = sorted(TEST_IMG.glob("*.jpg"))
per_scale: list = []

for i, sz in enumerate(IMGSZ_LIST):
    use_aug = AUGMENT and (not AUGMENT_LAST_ONLY or i == len(IMGSZ_LIST) - 1)
    rlist = list(
        infer_model.predict(
            source=[str(p) for p in test_paths],
            imgsz=sz,
            conf=CONF_INFER,
            iou=IOU_INFER,
            augment=use_aug,
            save=False,
            verbose=False,
            stream=True,
        )
    )
    if len(rlist) != len(test_paths):
        raise RuntimeError(f"Число результатов {len(rlist)} != {len(test_paths)}")
    per_scale.append(rlist)

print(
    "scales:", IMGSZ_LIST,
    "weights:", MS_WEIGHTS,
    "augment last only:", AUGMENT_LAST_ONLY,
    "n_test:", len(test_paths),
)


## 5. Постобработка и сабмит

- Формат Kaggle: `class score x1 y1 x2 y2` (пиксели; пусто → `0 0.000000 0 0 0 0`).
- `CONF_THR` — порог по confidence после мультимасштаба; `POST_NMS_IOU` — NMS по классам; `MAX_DET` — макс. число боксов на изображение.
- Без переобучения: подбор `CONF_THR`/`POST_NMS_IOU` по val + один сабмит; мультимасштаб даёт тот же «один прогон» но тяжелее **инференс**.


In [ ]:

CONF_THR = float(os.environ.get("CONF_THR", "0.10"))
POST_NMS_IOU = float(os.environ.get("POST_NMS_IOU", "0.55"))
MAX_DET = int(os.environ.get("MAX_DET", "120"))

rows = []
for j, p in enumerate(test_paths):
    preds = []
    for s in range(len(per_scale)):
        r = per_scale[s][j]
        xyxy, conf, cl = result_to_pred(r)
        conf, cl = apply_score_weights(conf, cl, MS_WEIGHTS[s])
        preds.append((xyxy, conf, cl))
    h, w = per_scale[0][j].orig_shape
    pred = merge_image_predictions(
        preds,
        float(w),
        float(h),
        CONF_THR,
        POST_NMS_IOU,
        MAX_DET,
    )
    rows.append({"image_id": p.stem, "PredictionString": pred})

pred_df = pd.DataFrame(rows)
template = pd.read_csv(SAMPLE_SUB)
sub = template[["image_id"]].merge(pred_df, on="image_id", how="left")
sub["PredictionString"] = sub["PredictionString"].fillna(EMPTY_PRED)
sub.loc[sub["PredictionString"].str.strip() == "", "PredictionString"] = EMPTY_PRED

out_path = ROOT / "sample_submission.csv"
sub.to_csv(out_path, index=False)
print("Saved:", out_path, "rows:", len(sub), "CONF_THR", CONF_THR, "POST_NMS_IOU", POST_NMS_IOU, "MAX_DET", MAX_DET)
sub.head()
